In [ ]:
import zipfile, os

zip_file_path = '/content/drive-download-20260612T140644Z-3-001.zip'
extract_dir = '/content/datasets/'

os.makedirs(extract_dir, exist_ok=True)

if os.path.exists(zip_file_path):
    print(f"Extracting {zip_file_path} to {extract_dir}...")
    with zipfile.ZipFile(zip_file_path, 'r') as z:
        z.extractall(extract_dir)
    print("Extraction complete.")
    print(f"Contents of {extract_dir}:")
    print(os.listdir(extract_dir))
else:
    print(f"Error: Zip file not found at {zip_file_path}.")
    print("Please ensure the zip file is uploaded or correctly mounted.")

In [ ]:
!git clone https://github.com/mlgig/FM4MTSC.git
!pip install momentfm --no-deps -q
!pip install einops -q

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
from sklearn.preprocessing import LabelEncoder
from momentfm import MOMENTPipeline

In [ ]:
data = np.load('/content/datasets/SYNTH.npy', allow_pickle=True)
print(type(data))
print(data.shape if hasattr(data, 'shape') else 'no shape')
if hasattr(data, 'item'):
    d = data.item()
    print(type(d))
    print(d.keys() if hasattr(d, 'keys') else d)

In [ ]:
datasets = {}
for name in ['SYNTH', 'CMJ', 'MP8', 'MP50']:
    data = np.load(f'/content/datasets/{name}.npy', allow_pickle=True).item()
    print(f'{name} train keys:', data['train'].keys())
    break

In [ ]:
datasets = {}
for name in ['SYNTH', 'CMJ', 'MP8', 'MP50']:
    data = np.load(f'/content/datasets/{name}.npy', allow_pickle=True).item()
    X_tr = torch.tensor(data['train']['X'], dtype=torch.float32)
    X_te = torch.tensor(data['test']['X'], dtype=torch.float32)
    le = LabelEncoder()
    y_tr = torch.tensor(le.fit_transform(data['train']['y']), dtype=torch.long)
    y_te = torch.tensor(le.transform(data['test']['y']), dtype=torch.long)
    datasets[name] = (X_tr, X_te, y_tr, y_te, le)
    print(f'{name}: X_train={X_tr.shape}, classes={le.classes_}')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using:', device)

results = {}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

name = 'SYNTH'
X_tr, X_te, y_tr, y_te, le = datasets[name]

model = MOMENTPipeline.from_pretrained(
    'AutonLab/MOMENT-1-large',
    model_kwargs={'task_name': 'classification', 'n_channels': X_tr.shape[1], 'num_class': len(le.classes_)}
)
model.init()
model = model.to(device)

for param in model.parameters():
    param.requires_grad = False
for param in model.head.parameters():
    param.requires_grad = True

loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
optimizer = Adam(model.head.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(2):
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        out = model(x_enc=X_batch)
        loss = criterion(out.logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1} loss: {total_loss/len(loader):.4f}')

torch.save(model.state_dict(), '/content/drive/MyDrive/MOMENT_SYNTH_trained.pt')
print('Saved!')

model.eval()
all_preds = []
with torch.no_grad():
    for i in range(0, len(X_te), 32):
        X_batch = X_te[i:i+32].to(device)
        out = model(x_enc=X_batch)
        preds = out.logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)

print(accuracy_score(y_te.numpy(), all_preds))
print(classification_report(y_te.numpy(), all_preds))

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

name = 'CMJ'
X_tr, X_te, y_tr, y_te, le = datasets[name]

model = MOMENTPipeline.from_pretrained(
    'AutonLab/MOMENT-1-large',
    model_kwargs={'task_name': 'classification', 'n_channels': X_tr.shape[1], 'num_class': len(le.classes_)}
)
model.init()
model = model.to(device)

for param in model.parameters():
    param.requires_grad = False
for param in model.head.parameters():
    param.requires_grad = True

loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
optimizer = Adam(model.head.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(2):
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        out = model(x_enc=X_batch)
        loss = criterion(out.logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1} loss: {total_loss/len(loader):.4f}')

torch.save(model.state_dict(), '/content/drive/MyDrive/MOMENT_CMJ_trained.pt')
print('Saved!')

model.eval()
all_preds = []
with torch.no_grad():
    for i in range(0, len(X_te), 32):
        X_batch = X_te[i:i+32].to(device)
        out = model(x_enc=X_batch)
        preds = out.logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)

print(accuracy_score(y_te.numpy(), all_preds))
print(classification_report(y_te.numpy(), all_preds))

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from momentfm import MOMENTPipeline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

datasets = {}
for name in ['MP8', 'MP50']:
    data = np.load(f'/content/datasets/{name}.npy', allow_pickle=True).item()
    X_tr = torch.tensor(data['train']['X'], dtype=torch.float32)
    X_te = torch.tensor(data['test']['X'], dtype=torch.float32)
    le = LabelEncoder()
    y_tr = torch.tensor(le.fit_transform(data['train']['y']), dtype=torch.long)
    y_te = torch.tensor(le.transform(data['test']['y']), dtype=torch.long)
    datasets[name] = (X_tr, X_te, y_tr, y_te, le)
    print(f'{name}: X_train={X_tr.shape}, classes={le.classes_}')

for name in ['MP8', 'MP50']:
    print(f'\n=== {name} ===')
    X_tr, X_te, y_tr, y_te, le = datasets[name]

    model = MOMENTPipeline.from_pretrained(
        'AutonLab/MOMENT-1-large',
        model_kwargs={'task_name': 'classification', 'n_channels': X_tr.shape[1], 'num_class': len(le.classes_)}
    )
    model.init()
    model = model.to(device)

    for param in model.parameters():
        param.requires_grad = False
    for param in model.head.parameters():
        param.requires_grad = True

    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
    optimizer = Adam(model.head.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(2):
        total_loss = 0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out = model(x_enc=X_batch)
            loss = criterion(out.logits, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'Epoch {epoch+1} loss: {total_loss/len(loader):.4f}')

    torch.save(model.state_dict(), f'/content/drive/MyDrive/MOMENT_{name}_trained.pt')
    print('Saved!')

    model.eval()
    all_preds = []
    with torch.no_grad():
        for i in range(0, len(X_te), 32):
            X_batch = X_te[i:i+32].to(device)
            out = model(x_enc=X_batch)
            preds = out.logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)

    acc = accuracy_score(y_te.numpy(), all_preds)
    print(f'Accuracy: {acc:.4f}')
    print(classification_report(y_te.numpy(), all_preds))